# QuantBench 模型适配器覆盖审计 v1

目标：审计 `src/model` 中 QuantBench 模型列表的显式适配器覆盖状态，确认不再存在未命名 `smoke` fallback，并复跑全模型矩阵回测，把 regime/window/qlib 风险分析 sidecar 固化到本实验目录。

本 notebook 是一次覆盖审计，不做模型上线选择；production 推广需要后续在更长窗口和更严格数据源上复验。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "SPEC.md").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("cannot locate repo root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "research" / "quantbench_model_coverage_audit_v1"
MATRIX_CARD = CARD / "matrix_card"
CARD.mkdir(parents=True, exist_ok=True)
MATRIX_CARD.mkdir(parents=True, exist_ok=True)

START = "2026-01-01"
END = "2026-06-09"
TRAIN_END = "2026-04-30"
CARD

PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1')

In [2]:
from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact
from src.model.artifacts import save_model_artifact
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.registry import QUANTBENCH_MODEL_NAMES, iter_model_specs
from src.model.training import model_adapter_name, model_adapter_status

len(QUANTBENCH_MODEL_NAMES), QUANTBENCH_MODEL_NAMES[:5], QUANTBENCH_MODEL_NAMES[-5:]

(32,
 ('logistic_clf',
  'ridge_ic',
  'ridge_mse',
  'random_forest_clf',
  'hist_gradient_mse'),
 ('rsr_mse', 'hypergraph_mse', 'thgnn_mse', 'sthan_mse', 'sthgcn_mse'))

In [3]:
fallback_expected: set[str] = set()
rows = []
for spec in iter_model_specs():
    rows.append(
        {
            "model": spec.name,
            "family": spec.family,
            "objective": spec.objective,
            "input_kind": spec.input_kind,
            "estimator": spec.estimator,
            "paper_fidelity": spec.paper_fidelity,
            "is_proxy": spec.paper_fidelity == "proxy_feature_adapter",
            "implementation_notes": spec.implementation_notes,
            "promotion_blocker": spec.promotion_blocker,
            "adapter_status": model_adapter_status(spec.name),
            "model_adapter": model_adapter_name(spec.name),
        }
    )

coverage = pd.DataFrame(rows)
smoke_status_models = coverage.loc[coverage["adapter_status"].eq("smoke_adapter"), "model"].tolist()
smoke_adapter_models = coverage.loc[coverage["model_adapter"].eq("smoke_fallback_v1"), "model"].tolist()
fallback_models = coverage.loc[coverage["adapter_status"].eq("dependency_fallback_adapter"), "model"].tolist()
strict_models = coverage.loc[coverage["adapter_status"].eq("paper_strict_adapter"), "model"].tolist()
fidelity_summary = model_fidelity_summary(QUANTBENCH_MODEL_NAMES)

assert smoke_status_models == []
assert smoke_adapter_models == []
assert set(fallback_models) == fallback_expected
assert set(strict_models) == set(QUANTBENCH_MODEL_NAMES)
assert fidelity_summary["model_count"] == len(QUANTBENCH_MODEL_NAMES)
assert fidelity_summary["fidelity_counts"]["proxy_feature_adapter"] > 0
assert coverage["implementation_notes"].astype(bool).all()

coverage_path = CARD / "adapter_coverage.csv"
coverage.to_csv(coverage_path, index=False)
coverage.groupby(["adapter_status", "paper_fidelity", "family"]).size().reset_index(name="count")

,adapter_status,paper_fidelity,family,count
0,paper_strict_adapter,native_library,boosting,3
1,paper_strict_adapter,native_library,linear,2
2,paper_strict_adapter,native_library,neural_tabular,1
3,paper_strict_adapter,native_library,tree,2
4,paper_strict_adapter,native_torch,sequence,5
5,paper_strict_adapter,proxy_feature_adapter,graph,9
6,paper_strict_adapter,proxy_feature_adapter,linear,1
7,paper_strict_adapter,proxy_feature_adapter,sequence,8
8,paper_strict_adapter,proxy_feature_adapter,tree,1


In [4]:
outputs = run_quantbench_matrix_smoke(
    card_dir=MATRIX_CARD,
    start=START,
    end=END,
    train_end=TRAIN_END,
    model_names=QUANTBENCH_MODEL_NAMES,
)

comparison = outputs.model_comparison.copy()
comparison_path = CARD / "matrix_model_comparison.csv"
comparison.to_csv(comparison_path, index=False)

assert int(outputs.predictions["model"].nunique()) == len(QUANTBENCH_MODEL_NAMES)
assert "smoke_adapter" not in set(outputs.predictions["adapter_status"])
assert "smoke_fallback_v1" not in set(outputs.predictions["model_adapter"])
assert set(outputs.summary["adapter_statuses"]) == {"paper_strict_adapter"}
assert outputs.summary["model_fidelity"]["schema_version"] == "model.fidelity.v1"
assert (MATRIX_CARD / "model_capabilities.csv").exists()

comparison.head(10)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,rank,model,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,1,gat_mse,67.157659,62.147247,5.010412,15.172143,6.090565,0.467756,5.351046,98.396039
1,2,mlp_mixer_mse,69.782463,62.147247,7.635216,17.132160,5.753609,0.512717,6.531791,92.839768
2,3,mlp_mse,71.522400,62.147247,9.375153,19.042370,5.467462,0.721769,6.239268,89.915524
3,4,dlinear_mse,55.947230,62.147247,-6.200016,15.379802,5.245452,0.524769,5.809831,77.557338
4,5,sequence_lag_ridge_mse,41.277413,62.147247,-20.869834,15.428993,4.895953,0.411463,4.909524,61.131650
5,6,lstm_mse,51.613406,62.147247,-10.533841,21.954412,3.976888,0.721407,2.807254,58.324763
6,7,hawkes_gru_mse,39.090644,62.147247,-23.056603,15.948241,4.205076,0.539885,2.852719,55.500664
7,8,patch_tst_mse,41.119229,62.147247,-21.028018,15.059349,4.619966,0.503862,6.782895,54.928914
8,9,random_forest_clf,52.839204,62.147247,-9.308043,25.150711,3.842480,0.752744,2.492709,54.840196
9,10,xgboost_mse,54.526208,62.147247,-7.621039,24.877561,4.055681,0.720331,4.628731,54.728602


In [5]:
price_frame = load_soxl_support_frame(start=START, end=END)
dataset = build_supervised_dataset(price_frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= TRAIN_END].copy()
test = dataset.frame.loc[dataset.frame["date"] > TRAIN_END].copy()

artifact_records = {}
expected_artifact_adapters = {
    "mlp_mse": "mlp_sklearn_mse_v1",
    "xgboost_mse": "xgboost_mse_v1",
    "catboost_rank": "catboost_rank_v1",
}
for model_name, expected_adapter in expected_artifact_adapters.items():
    artifact = fit_model_artifact(model_name, train=train, feature_columns=dataset.feature_columns[:8])
    predictions = artifact.predict(test)
    metadata = artifact.metadata_payload()

    artifact_path = save_model_artifact(CARD / f"{model_name}_artifact.joblib", artifact)
    predictions_path = CARD / f"{model_name}_artifact_predictions.csv"
    metadata_path = CARD / f"{model_name}_artifact_metadata.json"
    predictions.to_csv(predictions_path, index=False)
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

    assert metadata["adapter_status"] == "paper_strict_adapter"
    assert metadata["model_adapter"] == expected_adapter
    assert predictions["prediction"].notna().all()

    artifact_records[model_name] = {
        "artifact": artifact_path,
        "predictions": predictions_path,
        "metadata": metadata_path,
        "model_adapter": metadata["model_adapter"],
    }

artifact_records

{'mlp_mse': {'artifact': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1/mlp_mse_artifact.joblib'),
  'predictions': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1/mlp_mse_artifact_predictions.csv'),
  'metadata': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1/mlp_mse_artifact_metadata.json'),
  'model_adapter': 'mlp_sklearn_mse_v1'},
 'xgboost_mse': {'artifact': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1/xgboost_mse_artifact.joblib'),
  'predictions': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1/xgboost_mse_artifact_predictions.csv'),
  'metadata': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_model_coverage_audit_v1/xgboost_mse_artifact_metadata.json'),
  'model_adapter': 'xgboost_mse_v1'},
 'catboost_rank': {'artifact': PosixPath('/Users/rziiiii/dev/workspace/research/quantbench_mod

In [6]:
matrix_summary = json.loads((MATRIX_CARD / "analysis_summary.json").read_text(encoding="utf-8"))
audit_summary = {
    "status": "completed",
    "model_count": int(len(coverage)),
    "paper_strict_count": int((coverage["adapter_status"] == "paper_strict_adapter").sum()),
    "dependency_fallback_count": int((coverage["adapter_status"] == "dependency_fallback_adapter").sum()),
    "smoke_count": int((coverage["adapter_status"] == "smoke_adapter").sum()),
    "fallback_models": sorted(fallback_models),
    "model_fidelity": fidelity_summary,
    "matrix_status": matrix_summary["status"],
    "matrix_top_model": matrix_summary["top_model"],
    "train_end": TRAIN_END,
    "test_start": matrix_summary["test_start"],
    "test_end": matrix_summary["test_end"],
    "analysis_layers": matrix_summary["analysis_layers"],
    "artifacts": {
        "adapter_coverage": str(coverage_path.relative_to(CARD)),
        "matrix_model_comparison": str(comparison_path.relative_to(CARD)),
        "matrix_summary": str((MATRIX_CARD / "analysis_summary.json").relative_to(CARD)),
        "matrix_regime_metrics": str((MATRIX_CARD / "regime_metrics.csv").relative_to(CARD)),
        "matrix_window_metrics": str((MATRIX_CARD / "window_metrics.csv").relative_to(CARD)),
        "matrix_model_capabilities": str((MATRIX_CARD / "model_capabilities.csv").relative_to(CARD)),
        "matrix_qlib_risk_metrics": str((MATRIX_CARD / "qlib_risk_metrics.csv").relative_to(CARD)),
        "matrix_model_effect_deltas": str((MATRIX_CARD / "model_effect_deltas.csv").relative_to(CARD)),
        "model_artifacts": {
            model_name: {
                "artifact": str(record["artifact"].relative_to(CARD)),
                "predictions": str(record["predictions"].relative_to(CARD)),
                "metadata": str(record["metadata"].relative_to(CARD)),
                "model_adapter": record["model_adapter"],
            }
            for model_name, record in artifact_records.items()
        },
    },
    "risk_notes": [
        "xgboost_mse 与 catboost_rank 已接入原生依赖，但需要在 clean venv/CI 中持续验证 wheel 与 torch 的导入顺序稳定性。",
        "model_fidelity 已标记 proxy_feature_adapter；这些模型可训练可回测，但不等价于完整论文复现。",
        "本次窗口仅覆盖 2026-01-01 至 2026-06-09，不能直接外推到 production。",
    ],
}
(CARD / "analysis_summary.json").write_text(json.dumps(audit_summary, ensure_ascii=False, indent=2), encoding="utf-8")
audit_summary

{'status': 'completed',
 'model_count': 32,
 'paper_strict_count': 32,
 'dependency_fallback_count': 0,
 'smoke_count': 0,
 'fallback_models': [],
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 32,
  'fidelity_counts': {'native_library': 8,
   'native_torch': 5,
   'proxy_feature_adapter': 19},
  'proxy_models': ['autoformer_mse',
   'dlinear_mse',
   'extra_trees_rank',
   'fedformer_mse',
   'gat_mse',
   'gcn_mse',
   'hawkes_gru_mse',
   'hypergraph_mse',
   'market_graph_mse',
   'mlp_mixer_mse',
   'patch_tst_mse',
   'rgcn_mse',
   'ridge_ic',
   'rsr_mse',
   'sequence_lag_ridge_mse',
   'sfm_mse',
   'sthan_mse',
   'sthgcn_mse',
   'thgnn_mse'],
  'native_models': ['alstm_mse',
   'catboost_rank',
   'darnn_mse',
   'gru_mse',
   'hist_gradient_mse',
   'lightgbm_mse',
   'logistic_clf',
   'lstm_mse',
   'mlp_mse',
   'random_forest_clf',
   'ridge_mse',
   'tcn_mse',
   'xgboost_mse'],
  'promotion_blocker_count': 24},
 'matrix_status': 'smoke_c

## 结论

- 本次审计覆盖 QuantBench 注册表中的 32 个模型：32 个均为 `paper_strict_adapter`，`dependency_fallback_adapter` 和 `smoke_adapter` 数量均为 0。
- 新增 `model_fidelity` 分层：8 个 `native_library`、5 个 `native_torch`、19 个 `proxy_feature_adapter`；proxy 模型可训练可回测，但需要后续逐步替换为完整论文结构。
- `mlp_mse`、`xgboost_mse`、`catboost_rank` 均保存了可加载 artifact、预测和元数据；其中 `xgboost_mse` 使用 `xgboost_mse_v1`，`catboost_rank` 使用 `catboost_rank_v1`。
- 全模型矩阵已重跑，当前测试窗 top model 为 `gat_mse`；`matrix_card/` 下包含 `regime_metrics.csv`、`window_metrics.csv`、`model_capabilities.csv`、`qlib_risk_metrics.csv` 和 `model_effect_deltas.csv`，用于继续比较不同区间和 regime 的模型效果差异。
- 结论不建议直接推广 production：boosting 依赖已接入但仍需 clean venv/CI 复验，19 个 proxy adapter 需要后续按论文 fidelity 逐步替换。下一步应优先处理 regime-aware model selection 审计和 proxy 替换优先级。